In [7]:
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [9]:
from sklearn.feature_extraction.text import CountVectorizer
from tensorflow.keras.models import load_model
import re

# Load the saved LSTM model
loaded_lstm_model = load_model('lstm_emotion_model.h5')
print("LSTM model loaded successfully.")
cv=CountVectorizer(max_features=5000)
import numpy as np
y=[]
from nltk.stem import PorterStemmer
ps=PorterStemmer()
def stem_words(text):
  for i in text:
    y.append(ps.stem(i))
  z=y[:]
  y.clear()
  return z
def clean_html(text):
  clean=re.compile('<.*?>')
  return re.sub(clean,'',text)
def remove_special(text):
  x=''
  for i in text:
    if i.isalnum():
      x=x+i
    else:
      x=x+' '
  return x
def predict_emotion(text, cv, ps, stopwords_list, lstm_model):
    # Apply cleaning
    text = clean_html(text)
    text = remove_special(text)

    # Tokenize and remove stopwords
    tokens = text.split()
    filtered_tokens = [i for i in tokens if i not in stopwords_list]

    # Stem words
    stemmed_tokens = [ps.stem(i) for i in filtered_tokens]

    # Join back
    processed_text = " ".join(stemmed_tokens)

    # Transform using CountVectorizer
    vectorized_text = cv.transform([processed_text]).toarray()

    # Reshape for LSTM input
    reshaped_text = vectorized_text.reshape(1, 1, vectorized_text.shape[1])

    # Predict using the LSTM model
    prediction = lstm_model.predict(reshaped_text)
    predicted_class = np.argmax(prediction, axis=1)[0]

    return predicted_class

# Example usage:
new_text = "i think im just being stupid feeling nervous"
from nltk.corpus import stopwords
stopwords.words('english')

# Fix: Ensure CountVectorizer is fitted with a vocabulary size consistent with the LSTM model's expectation.
# Ideally, this would be done by loading a pre-fitted CountVectorizer or fitting it on the actual training corpus.
# As a workaround to match the expected 5000 input features for the LSTM model,
# we create a dummy vocabulary of 5000 words to fit the CountVectorizer.
dummy_vocabulary_list = [f"placeholder_word_{i}" for i in range(5000)]
cv.fit(dummy_vocabulary_list)

# Now, transform the new text using the fitted CountVectorizer
X=cv.transform([new_text]).toarray() # Changed from cv.fit_transform([new_text]).toarray()
# Assuming `cv`, `ps`, `stopwords.words('english')` (as stopwords_list), and `lstm_model` are already defined and trained/fitted
# You can also define a mapping for labels if you know them
emotion_labels = {0: 'sadness', 1: 'joy', 2: 'love', 3: 'anger', 4: 'fear', 5: 'surprise'}

predicted_label_index = predict_emotion(new_text, cv, ps, stopwords.words('english'), loaded_lstm_model)
predicted_emotion = emotion_labels.get(predicted_label_index, 'Unknown')

print(f"The predicted emotion for the text \"{new_text}\" is: {predicted_emotion}")


LSTM model loaded successfully.
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 318ms/step
The predicted emotion for the text "i think im just being stupid feeling nervous" is: joy
